In [1]:
import pandas as pd
import os
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import rcParams

rcParams['font.family'] = 'SimHei'

In [12]:
# 异常值滤除了 2分钟数据
# 可视化阶段，bupt_maydat.csv, may_filtered.csv 都是可用的

In [14]:
import numpy as np

t = np.load('may_time.npy')
# Convert to int64 (seconds since epoch) for easier calculation
t_sec = t.astype('datetime64[s]').astype('int64')

# Calculate differences
diffs = np.diff(t_sec)
print('Diffs stats: min=', diffs.min(), 'max=', diffs.max(), 'mean=', diffs.mean())

# Find gaps > 1 second
gap_mask = diffs > 1
gap_indices = np.where(gap_mask)[0]
print('Number of gaps (>1s):', len(gap_indices))
print('Gap locations (indices in diffs):', gap_indices[:20])

# Segment boundaries: split at each gap
# segments are defined by gap_indices: segment i is [gap_indices[i-1]+1, gap_indices[i]] if i>0 else [0, gap_indices[0]]
segment_starts = [0] + (gap_indices + 1).tolist()
segment_ends   = gap_indices.tolist() + [len(t_sec) - 1]

durations = []
for s, e in zip(segment_starts, segment_ends):
    dur = t_sec[e] - t_sec[s]
    durations.append(dur)

durations = np.array(durations)
print('Number of segments:', len(durations))
print('Total duration (sum of segments):', durations.sum(), 'seconds')
print('Overall span:', t_sec[-1] - t_sec[0], 'seconds')
ratio = durations.sum() / (t_sec[-1] - t_sec[0])
print('Continuous time ratio:', round(ratio, 4), f'({ratio*100:.2f}%)')
print('Longest segment:', durations.max(), 'seconds')
print('Shortest segment:', durations.min(), 'seconds')
print('Segment durations (first 10):', durations[:10])


Diffs stats: min= 1 max= 11255 mean= 1.2806130723901172
Number of gaps (>1s): 214
Gap locations (indices in diffs): [  184   712  4161  4165  4174  4505  5544  5550  5576  5628  5653  5683
  5733  5746  5777  5904  7958 10998 11793 13764]
Number of segments: 215
Total duration (sum of segments): 73774 seconds
Overall span: 94750 seconds
Continuous time ratio: 0.7786 (77.86%)
Longest segment: 13191 seconds
Shortest segment: 0 seconds
Segment durations (first 10): [ 184  527 3448    3    8  330 1038    5   25   51]


In [16]:
time = np.load("may_time.npy")
time

array(['2026-05-02T16:21:44', '2026-05-02T16:21:45',
       '2026-05-02T16:21:46', ..., '2026-05-03T18:40:52',
       '2026-05-03T18:40:53', '2026-05-03T18:40:54'],
      dtype='datetime64[s]')

In [17]:
t = time.astype('int64')
t

array([1777738904, 1777738905, 1777738906, ..., 1777833652, 1777833653,
       1777833654], dtype=int64)

In [18]:
diffs = np.diff(t)

In [19]:
diffs

array([1, 1, 1, ..., 1, 1, 1], dtype=int64)

In [33]:
diffs[diffs > 10].shape

(100,)

In [34]:
al = pd.read_csv("aligned_features.csv")
al

,timestamp,rri_ms,asc_area,desc_area,motion
0,2026-05-02 16:21:46,1150.0,310.0,2478.0,1.0
1,2026-05-02 16:21:48,810.0,517.0,983.0,1.0
2,2026-05-02 16:21:49,770.0,537.0,1034.0,1.0
3,2026-05-02 16:21:49,650.0,453.0,1216.0,1.0
4,2026-05-02 16:21:50,780.0,329.0,1288.0,1.0
...,...,...,...,...,...
38010,2026-05-03 18:40:27,620.0,190.0,1444.0,4.0
38011,2026-05-03 18:40:27,720.0,1580.0,354.0,6.0
38012,2026-05-03 18:40:33,1390.0,1763.0,368.0,5.0
38013,2026-05-03 18:40:34,1420.0,1089.0,3195.0,8.0


In [36]:
time = al.timestamp.astype('datetime64[s]').astype('int64')

In [37]:
time

0        1777738906
1        1777738908
2        1777738909
3        1777738909
4        1777738910
            ...    
38010    1777833627
38011    1777833627
38012    1777833633
38013    1777833634
38014    1777833639
Name: timestamp, Length: 38015, dtype: int64

In [39]:
di = np.diff(time)

In [42]:
gap_mask = di > 1
gap_indices = np.where(gap_mask)[0]
gap_indices

array([    0,     5,     8, ..., 38009, 38011, 38013], dtype=int64)

In [43]:
len(gap_indices)

11503

In [44]:
segment_starts = [0] + (gap_indices + 1).tolist()
segment_ends   = gap_indices.tolist() + [len(time) - 1]

durations = []
for s, e in zip(segment_starts, segment_ends):
    dur = time[e] - time[s]
    durations.append(dur)

durations = np.array(durations)

print('Total duration (sum of segments):', durations.sum(), 'seconds')


Total duration (sum of segments): 21036 seconds


In [45]:
# 处理后的戒指特征只有 5.8 h

In [2]:
watch = pd.read_csv("watch_features.csv")
watch

,record_time,sub_index,rri,area_up,area_down,motion
0,2024-12-31 22:06:41,0,880,726,1928,910
1,2024-12-31 22:06:41,1,990,723,1915,407
2,2024-12-31 22:06:42,0,760,720,1903,42
3,2024-12-31 22:06:43,0,570,717,1890,7
4,2024-12-31 22:06:44,0,750,715,1878,23
...,...,...,...,...,...,...
94843,2025-01-01 20:58:17,0,870,1140,1972,3
94844,2025-01-01 20:58:18,0,960,1139,1972,2
94845,2025-01-01 20:58:19,0,950,1139,1971,1
94846,2025-01-01 20:58:20,0,950,1139,1971,0


In [3]:
time = watch.record_time.astype("datetime64[s]").astype("int64")

In [4]:
diffs = np.diff(time)
diffs

array([0, 1, 1, ..., 1, 1, 1], dtype=int64)

In [5]:
gap_mask = diffs > 1
gap_indices = np.where(gap_mask)[0]


In [6]:
segment_starts = [0] + (gap_indices + 1).tolist()

In [7]:
segment_ends   = gap_indices.tolist() + [len(time) - 1]
segment_ends[:5]

[5663, 18431, 25216, 25535, 33958]

In [8]:
durations = []
for s, e in zip(segment_starts, segment_ends):
    dur = time[e] - time[s]
    durations.append(dur)

durations = np.array(durations)

print('Total duration (sum of segments):', durations.sum(), 'seconds')

Total duration (sum of segments): 79676 seconds


In [10]:
len(segment_starts)

29

In [13]:
# 某次手表特征数据的时长是22.86
h, 间断时长是43min，连续片段总长：22.1h

In [12]:
time.max() - time.min()

82300